# 🤖 Fine-Tuning BERT on IMDB Movie Reviews Dataset
## Data Science Internship – February 2026 | Task 4: NLP

---

## 📌 Objective
The goal of this assignment is to fine-tune a pre-trained **BERT (Bidirectional Encoder Representations from Transformers)** model on the **IMDB Movie Reviews** dataset for **binary sentiment classification** (Positive / Negative).

---

## 📖 What is BERT?

**BERT** is a transformer-based model developed by **Google AI** in 2018.  
It reads text in **both directions** simultaneously using a Transformer Encoder architecture.

### 🔑 Key Concepts:
- **Bidirectional**: Reads text left-to-right AND right-to-left at the same time
- **Pre-training Tasks**:
  - **Masked Language Modeling (MLM)**: Randomly masks words and predicts them
  - **Next Sentence Prediction (NSP)**: Predicts if sentence B follows sentence A
- **Fine-Tuning**: BERT is adapted to specific tasks like text classification

---

## 🏗️ BERT Architecture

| Component       | Details              |
|----------------|----------------------|
| Model Type      | Transformer Encoder  |
| Layers          | 12 (BERT-base)       |
| Hidden Size     | 768                  |
| Attention Heads | 12                   |
| Parameters      | ~110 Million         |
| Vocabulary Size | 30,522 tokens        |

---

## 🔄 Pipeline Overview

Raw Text → Preprocessing → Tokenization → Model Training → Evaluation → Experiments → Conclusion

## 🔧 Step 1: Install Required Libraries

### Theory:
We need the following libraries:

| Library | Purpose |
|--------|---------|
| `transformers` | Hugging Face library — provides BERT model & tokenizer |
| `datasets` | Load IMDB dataset directly from Hugging Face |
| `torch` | PyTorch — deep learning framework for training |
| `scikit-learn` | Evaluation metrics (accuracy, F1, confusion matrix) |
| `seaborn` | Visualization of confusion matrix |
| `matplotlib` | Plotting graphs and charts |
| `tqdm` | Progress bar during training |

In [34]:
# Install all required libraries
!pip install transformers datasets torch scikit-learn seaborn matplotlib tqdm -q

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


## 📦 Step 2: Import Libraries

### Theory:
We import all necessary modules:
- **numpy, pandas**: Data handling and manipulation
- **torch**: PyTorch for model training
- **transformers**: BERT model and tokenizer
- **datasets**: Load IMDB dataset
- **sklearn**: Evaluation metrics
- **matplotlib, seaborn**: Visualization
- **re**: Regular expressions for text cleaning
- **tqdm**: Progress bar during training loops

We also set a **random seed = 42** for reproducibility and check if **GPU** is available.

In [8]:
# =============================================
# Import All Required Libraries
# =============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Hugging Face Datasets
from datasets import load_dataset

# Scikit-learn metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Progress bar
from tqdm import tqdm

# -----------------------------------------------
# Set random seed for reproducibility
# -----------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------
# Check GPU availability
# -----------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ All libraries imported successfully!")
print(f"🖥️  Device in use: {device}")

ImportError: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _dataset: The specified module could not be found.)

## 📥 Step 3: Load IMDB Dataset

### Theory: IMDB Movie Reviews Dataset
The **IMDB Movie Reviews** dataset is a classic NLP benchmark dataset:

| Property | Details |
|----------|---------|
| Total Samples | 50,000 reviews |
| Train Set | 25,000 reviews |
| Test Set | 25,000 reviews |
| Classes | Positive (1) / Negative (0) |
| Balance | Perfectly balanced (50-50) |

We use Hugging Face's `datasets` library to load it directly.  
No manual download needed!

In [32]:
# =============================================
# Load IMDB Dataset
# =============================================

print("📥 Loading IMDB dataset...")

# Load dataset from Hugging Face
dataset = load_dataset("IMDB Dataset")

# Convert to pandas DataFrames
train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

print(f"✅ Dataset loaded!")
print(f"📊 Train size : {len(train_df)}")
print(f"📊 Test size  : {len(test_df)}")
print(f"\n🔍 Sample Data:")
print(train_df.head(3))

📥 Loading IMDB dataset...


NameError: name 'load_dataset' is not defined

## 🧹 Step 4: Data Preprocessing

### Theory: Why Preprocessing?
Raw IMDB text contains noise that hurts model performance:

| Issue | Example | Fix |
|-------|---------|-----|
| HTML tags | `<br />`, `<p>` | Remove with regex |
| Special characters | `@, #, $, %` | Remove non-alphabetic chars |
| Extra whitespace | `"hello   world"` | Normalize spaces |
| Uppercase letters | `"GREAT movie"` | Lowercase (bert-base-uncased) |

### Steps:
1. Remove HTML tags using `re.sub()`
2. Remove special characters
3. Convert to lowercase
4. Strip extra whitespace
5. Handle missing values

In [14]:
# =============================================
# Data Preprocessing
# =============================================

def clean_text(text):
    """
    Cleans raw review text:
    - Removes HTML tags
    - Removes special characters
    - Converts to lowercase
    - Normalizes whitespace
    """
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    
    # Remove special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# -----------------------------------------------
# Check for missing values
# -----------------------------------------------
print("🔍 Checking for missing values...")
print(f"Train:\n{train_df.isnull().sum()}")
print(f"\nTest:\n{test_df.isnull().sum()}")

# -----------------------------------------------
# Apply cleaning
# -----------------------------------------------
print("\n🧹 Cleaning text data...")
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text']  = test_df['text'].apply(clean_text)

# Show before/after example
print("\n📝 Before Cleaning:")
print(train_df['text'].iloc[0][:200])
print("\n📝 After Cleaning:")
print(train_df['cleaned_text'].iloc[0][:200])

print("\n✅ Preprocessing complete!")

🔍 Checking for missing values...


NameError: name 'train_df' is not defined